In [82]:
import os
import glob
import cv2
import shutil
import py7zr
import numpy as np
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd

from pathlib import Path

In [83]:
def apply_mask(frame, mask):
  """
    ``apply_mask`` applies a the binary mask `mask` to the given `frame` of a video.

    :param frame: frame of a video in `cv2.COLOR_BGR2GRAY` format
    :param mask: binary mask with same size as `frame`
    :return: returns the frame image with the mask applied
    """ 
  return cv2.bitwise_and(frame, frame, mask=mask)

def remove_black_frame(sum_image, top = 3, bottom = 30, left = 5, right = 12):
    h, w = sum_image.shape
    return sum_image[top : h - bottom, left : w - right]


def generate_sum_image(img_input_path, xml_input_path, output_path, mask_type="folgueroles"):

    # STEP 1: Read the video -----------------------------------------------------------------
    cap = cv2.VideoCapture(img_input_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video {img_input_path}")
    
    # ---- Read the properties
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)

    # STEP 2: Define a mask ------------------------------------------------------------------
    my_mask = np.zeros((height, width), dtype=np.uint8)
    if mask_type == "folgueroles":
        mask_height = height - 30
        cv2.rectangle(my_mask, (0, 0), (width, mask_height), 255, thickness=-1)
    elif mask_type == "none":
        cv2.rectangle(my_mask, (0, 0), (width, height), 255, thickness=-1)
    else:
        raise ValueError("Invalid mask_type")

    # STEP 3: Read first frame and perpare background ----------------------------------------
    ret, first_frame = cap.read()
    if not ret:
        raise IOError("Cannot read first frame")
    
    first_frame_gray = cv2.cvtColor(first_frame, cv2.COLOR_BGR2GRAY)
    masked_first_frame = apply_mask(first_frame_gray, my_mask)

    # STEP 4: Process frames -----------------------------------------------------------------
    frame_index = 0

    # Sum-images and (optional) stacks initialization
    sum_image = np.zeros_like(masked_first_frame, dtype=np.uint8)
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_index += 1    # Increment frame that we are processing

        current_frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)    # Grayscaled first frame
        current_frame_masked = apply_mask(current_frame_gray, my_mask)
        frame_diff = cv2.absdiff(current_frame_masked, masked_first_frame)    # Subtract first frame to remove background
        sum_image = np.maximum(sum_image, frame_diff)     # Add frame to the sum-image (getting the max value per pixel)

    sum_image = remove_black_frame(sum_image)
    out_path = Path(output_path) / f"{Path(img_input_path).stem}_SUMIMG.png"
    cv2.imwrite(str(out_path), sum_image)

    cap.release()

    return sum_image

def get_bbox_metadata(input_path, padding=32):

    # Open XML element
    tree = ET.parse(input_path)
    root = tree.getroot()

    # Get video data
    width = float(root.attrib['cx'])
    height = float(root.attrib['cy'])
    frames = float(root.attrib['frames'])
    fps = float(root.attrib['fps'])

    # Get event occurrance time
    year = float(root.attrib['y'])
    month = float(root.attrib['mo'])
    day = float(root.attrib['d'])
    hour = float(root.attrib['h'])
    minute = float(root.attrib['m'])

    # Get location data
    lng = float(root.attrib['lng'])
    lat = float(root.attrib['lat'])
    alt = float(root.attrib['alt'])
    camera = str(root.attrib['sid'])

    # Get all uc_path elements
    path_points = root.find('ufocapture_paths')

    x_vals = [float(p.attrib['x']) for p in path_points.findall('uc_path')]
    y_vals = [float(p.attrib['y']) for p in path_points.findall('uc_path')]
    brightnes_vals = [float(p.attrib['bmax']) for p in path_points.findall('uc_path')]
    frame_nums = [float(p.attrib['fno']) for p in path_points.findall('uc_path')]

    # Compute bounding box
    bbox = {
        'x_min': max(0, min(x_vals) - padding),
        'x_max': min(width, max(x_vals) + padding),
        'y_min': max(0, min(y_vals) - padding),
        'y_max': min(height, max(y_vals) + padding)
    }

    # Compute time and brightness
    time_seconds = (frame_nums[-1] - frame_nums[0] + 1) / fps
    mean_brightness = np.mean(brightnes_vals)

    metadata = {
        "filename": Path(input_path).stem,
        "year": year,
        "month": month,
        "day": day,
        "hour": hour,
        "minute": minute,
        "lng": lng,
        "lat": lat,
        "alt": alt,
        "camera": camera,
        "width": bbox['x_max'] - bbox['x_min'],
        "height": bbox['y_max'] - bbox['y_min'],
        "frames": frames,
        "fps": fps,
        "time": time_seconds,
        "bmin": min(brightnes_vals),
        "bmax": max(brightnes_vals),
        "mean_brightness": mean_brightness
    }

    return bbox, metadata

def generate_cropped_sum_image(sum_img, img_input_path, xml_input_path, output_path, padding=64,
                               top=3, left=5):

    # Get bounding box and crop sum-image
    bbox, metadata = get_bbox_metadata(input_path=xml_input_path, padding=padding)

    x_min, x_max = int(bbox['x_min']), int(bbox['x_max'])
    y_min, y_max = int(bbox['y_min']), int(bbox['y_max'])

    # Adjust coordinates to cropped image (black frame removed)
    # Clamp to valid range to avoid negative indices
    h, w = sum_img.shape
    x_min_adj = max(0, x_min - left)
    x_max_adj = min(w, x_max - left)
    y_min_adj = max(0, y_min - top)
    y_max_adj = min(h, y_max - top)

    cropped_sum_img = sum_img[y_min_adj:y_max_adj, x_min_adj:x_max_adj]

    out_path = Path(output_path) / f"{Path(img_input_path).stem}_CROP_SUMIMG.png"
    cv2.imwrite(str(out_path), cropped_sum_img)

    return cropped_sum_img, bbox, metadata

In [84]:
sum_img = generate_sum_image("M20260101_000555_MasLaRoca_NE.avi", "M20260101_000555_MasLaRoca_NE.xml", "./")

In [85]:
import matplotlib.pyplot as plt 

def plot_img(img):
    plt.figure(figsize=(8, 6))
    plt.imshow(img, cmap='gray') 
    plt.title("Sum Image") 
    plt.axis('off') 
    plt.show()

In [86]:
top = 3
bottom = 30
left = 5
right = 12

height, width = sum_img.shape
crop = sum_img[top : height - bottom,
                 left : width - right]
cv2.imwrite("./crop.png", crop)

True

In [87]:
sum_image_cropped, bbox, metadata = generate_cropped_sum_image(sum_img=crop,
                                                                            img_input_path="./crop.png", 
                                                                            xml_input_path="M20260101_000555_MasLaRoca_NE.xml", 
                                                                            output_path=f"./sum_image_cropped")

In [88]:
print(metadata)

{'filename': 'M20260101_000555_MasLaRoca_NE', 'year': 2026.0, 'month': 1.0, 'day': 1.0, 'hour': 0.0, 'minute': 5.0, 'lng': 2.19, 'lat': 41.56, 'alt': 570.0, 'camera': 'NE', 'width': 254.2, 'height': 83.70000000000005, 'frames': 82.0, 'fps': 25.0, 'time': 0.84, 'bmin': 38.0, 'bmax': 62.0, 'mean_brightness': np.float64(47.0)}


In [89]:
cv2.imwrite("./cropped1.png", sum_image_cropped)

True